In [1]:
# %pip install qiskit==1.2.4
# %pip install qiskit-aer==0.15.1
# %pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# BB84 (no attacker)

This notebook simulates the BB84 quantum key distribution protocol with two
participants, **Alice** and **Bob**. All random choices are made by measuring
the superposition state $(|0\rangle + |1\rangle)/\sqrt{2}$ — no classical RNG
is used. With no eavesdropper, the sifted bits Alice and Bob hold should agree
perfectly, so the measured error rate on the public sample is expected to be 0
and the final keys should be identical.

In [2]:
N = 100              # number of qubits Alice transmits
THRESHOLD = 0.11     # eavesdropping detection threshold 

backend = BasicSimulator()

## Alice prepares
Alice generates her secret bits and her basis choices.

In [3]:
# Alice generates N random bits by measuring |+> once each.
alice_bits = []
for _ in range(N):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    alice_bits_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    alice_bits.append(int(alice_bits_result))

# Alice generates N random basis choices the same way.
alice_bases = []
for _ in range(N):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    alice_base_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    alice_bases.append(int(alice_base_result))

print("Alice's bits   :", "".join(str(b) for b in alice_bits))
print("Alice's bases  :", "".join(str(b) for b in alice_bases))
print("\nbasis bit 0 -> standard basis {|0>, |1>}\nbasis bit 1 -> diagonal basis {|+>, |->}")


Alice's bits   : 1111100001011100010111011111111000001111110100000010011101111011010000111100111110000110010100100111
Alice's bases  : 1001011101010111011000101101110111010000111111001101000011011010100101011001001111000111000110001000

basis bit 0 -> standard basis {|0>, |1>}
basis bit 1 -> diagonal basis {|+>, |->}


## Bob chooses bases
Bob picks a random measurement basis for every incoming qubit.

In [4]:
# Bob, independently and in advance, chooses a random basis for every incoming qubit.
bob_bases = []
for _ in range(N):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    bob_base_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    bob_bases.append(int(bob_base_result))

print("Bob's bases    :", "".join(str(b) for b in bob_bases))
print("\nbasis bit 0 -> standard basis {|0>, |1>}\nbasis bit 1 -> diagonal basis {|+>, |->}")

Bob's bases    : 0110101000011110011010001001101000010000000110100110010000010011001101000100011110000101010011110001

basis bit 0 -> standard basis {|0>, |1>}
basis bit 1 -> diagonal basis {|+>, |->}


## Quantum transmission and Bob's measurement
One circuit per qubit: Alice's preparation gates followed by Bob's measurement.

In [5]:
# For each of the N transmissions we build a single 1-qubit circuit:
#   Alice's preparation gates  ->  Bob's measurement gates  ->  measure.
bob_bits = []
for i in range(N):
    qc = QuantumCircuit(1, 1)

    # --- Alice prepares qubit i ---
    if alice_bits[i] == 1:
        qc.x(0)                  # encode value bit
    if alice_bases[i] == 1:
        qc.h(0)                  # rotate into diagonal basis if needed

    # --- Bob measures qubit i ---
    if bob_bases[i] == 1:
        qc.h(0)                  # rotate diagonal basis back to computational before measuring
    qc.measure(0, 0)

    bob_bits_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    bob_bits.append(int(bob_bits_result))

print("Bob's bits     :", "".join(str(b) for b in bob_bits))

Bob's bits     : 1000000001010100010111011111110110001111000100000011001110110011011000110101101110000110000101000110


## Basis reconciliation (sifting)
Alice and Bob keep only the positions where their bases matched.

In [6]:
# Alice and Bob publicly announce their bases and keep only the positions where they agree.
basis_match = [alice_bases[i] == bob_bases[i] for i in range(N)]
sifted_alice = [alice_bits[i] for i in range(N) if basis_match[i]]
sifted_bob = [bob_bits[i] for i in range(N) if basis_match[i]]

print("Basis match    :", "".join("1" if match else "0" for match in basis_match))
print("Sifted length  :", len(sifted_alice))
print("Sifted Alice   :", "".join(str(bits) for bits in sifted_alice))
print("Sifted Bob     :", "".join(str(bits) for bits in sifted_bob))

Basis match    : 0000001010110110111101011011100000111111000110010100101100110110010111100010101110111101101010000110
Sifted length  : 53
Sifted Alice   : 00011001011111110011111000011110110001011110001000011
Sifted Bob     : 00011001011111110011111000011110110001011110001000011


## Eavesdropping check
A random half of the sifted key is publicly compared. The mismatch rate on this sample estimates the channel error rate.

In [7]:
# Eavesdropping check: for each position in the sifted key we randomly choose whether to sample it.
#   result == 1 -> the bit is publicly compared.
#   result == 0 -> the bit is kept as part of the final secret key.
sample_mask = []
for _ in range(len(sifted_alice)):
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    mask_result = backend.run(transpile(qc, backend), shots=1, memory=True).result().get_memory()[0]
    sample_mask.append(int(mask_result))

sample_indices = [i for i, mask in enumerate(sample_mask) if mask == 1]
sample_alice   = [sifted_alice[i] for i in sample_indices]
sample_bob     = [sifted_bob[i]   for i in sample_indices]

mismatch_count = sum(1 for a, b in zip(sample_alice, sample_bob) if a != b)
error_rate     = mismatch_count / len(sample_alice) if sample_alice else 0.0

print("Sample indices :", sample_indices)
print("Sample Alice   :", "".join(str(b) for b in sample_alice))
print("Sample Bob     :", "".join(str(b) for b in sample_bob))
print(f"Mismatches     : {mismatch_count} / {len(sample_alice)}")
print(f"Error rate     : {error_rate:.3f}  (threshold = {THRESHOLD})")

Sample indices : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 17, 19, 21, 23, 25, 26, 27, 28, 35, 36, 37, 38, 42, 44, 46, 48, 50, 51]
Sample Alice   : 00011001011011000110010101001
Sample Bob     : 00011001011011000110010101001
Mismatches     : 0 / 29
Error rate     : 0.000  (threshold = 0.11)


## Final key and verdict
Unsampled sifted bits form the shared key. Without an attacker we expect a 0% error rate and identical final keys.

In [8]:
final_key_alice = [sifted_alice[i] for i, mask in enumerate(sample_mask) if mask == 0]
final_key_bob   = [sifted_bob[i]   for i, mask in enumerate(sample_mask) if mask == 0]

print("Final key Alice:", "".join(str(bits) for bits in final_key_alice))
print("Final key Bob  :", "".join(str(bits) for bits in final_key_bob))
print("Keys match     :", final_key_alice == final_key_bob)
print("Final key len  :", len(final_key_alice))

if error_rate > THRESHOLD:
    print(f"\nALARM: error rate {error_rate:.3f} exceeds threshold {THRESHOLD}.")
    print("An eavesdropper is likely present -- ABORT and discard the key.")
else:
    print(f"\nOK: error rate {error_rate:.3f} is within threshold {THRESHOLD}.")
    print("No eavesdropping detected -- final key accepted.")

Final key Alice: 111110111011011011100001
Final key Bob  : 111110111011011011100001
Keys match     : True
Final key len  : 24

OK: error rate 0.000 is within threshold 0.11.
No eavesdropping detected -- final key accepted.
